In [ ]:
experiment_results = []

for model_name, configurations in experiments.items():

    for model in configurations:

        if model_name == "Decision Tree":
            condition = f"Depth {model.max_depth}"

        else:
            condition = f"{model.n_estimators} Trees"

        start_time = time.time()
        model.fit(X_train, y_train)
        training_time = time.time() - start_time

        train_prediction = model.predict(X_train)
        test_prediction = model.predict(X_test)

        cv_scores = cross_val_score(
            model, X_train, y_train,
            cv=5, scoring="r2"
        )

        experiment_results.append({
            "Model": model_name,
            "Condition": condition,
            "MAE": mean_absolute_error(
                y_test, test_prediction
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    y_test, test_prediction
                )
            ),
            "Test R2": r2_score(
                y_test, test_prediction
            ),
            "Train R2": r2_score(
                y_train, train_prediction
            ),
            "CV Mean R2": cv_scores.mean(),
            "CV Std R2": cv_scores.std(),
            "Training Time (sec)": training_time
        })

experiment_df = pd.DataFrame(experiment_results)

print("\nEXPERIMENTAL CONDITIONS")
print(experiment_df.round(4).to_string(index=False))

In [ ]:
complexity_results = []

# Linear Regression
linear_model = models["Linear Regression"].named_steps["model"]

complexity_results.append({
    "Model": "Linear Regression",
    "Parameters": len(linear_model.coef_),
    "Depth": np.nan,
    "Nodes": np.nan,
    "Trees": np.nan
})

# Decision Tree
tree_model = models["Decision Tree"]

complexity_results.append({
    "Model": "Decision Tree",
    "Parameters": np.nan,
    "Depth": tree_model.get_depth(),
    "Nodes": tree_model.tree_.node_count,
    "Trees": 1
})

# Random Forest
forest_model = models["Random Forest"]

total_nodes = sum(
    tree.tree_.node_count
    for tree in forest_model.estimators_
)

average_depth = np.mean([
    tree.tree_.max_depth
    for tree in forest_model.estimators_
])

complexity_results.append({
    "Model": "Random Forest",
    "Parameters": np.nan,
    "Depth": average_depth,
    "Nodes": total_nodes,
    "Trees": forest_model.n_estimators
})

complexity_df = pd.DataFrame(complexity_results)

print("\nMODEL COMPLEXITY")
print(complexity_df.round(2).to_string(index=False))